[![../img/pythonista.png](../img/pythonista.png)](https://www.pythonista.io)

# *NumPy* y *pandas* esenciales, tipos de variables.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete
**Lunes 19 de octubre de 2026 · 6:30 a 8:30 pm (CDMX)** · Módulo I: *Datos y su calidad*

> 🔑 **Versión con soluciones.** Solo para el instructor o para revisión posterior.

En esta sesión se repasarán los fundamentos de *NumPy* y *pandas*, se clasificarán las variables por su tipo y por la forma en que se recolectaron, y se presentarán los cambios de *pandas* 3 que afectan al resto del curso.

## 🎯 Objetivo de la sesión.

Cargar y explorar un *dataframe* con soltura, y **clasificar cada variable por su tipo y por cómo se recolectó**.

Al terminar podrás:
1. Usar arreglos de *NumPy*: `dtype`, operaciones vectorizadas, *broadcasting* y máscaras booleanas.
2. Leer y explorar datos con *pandas* 3: `info`, `describe`, `loc`/`iloc`, filtros y `assign`.
3. Distinguir variables numéricas (continuas/discretas), categóricas (nominales/ordinales) y fechas, y representarlas con el `dtype` correcto (`str`, `category`, tipos *nullable*).
4. Reconocer sesgos de recolección: selección, supervivencia, no respuesta y medición.

## 🗺️ Agenda (120 min).

| Min | Bloque |
|---|---|
| 10 | Bienvenida, reglas de evaluación y asistencia, arranque del *Codespace* |
| 25 | 1. *NumPy* esencial |
| 35 | 2. *pandas* esencial (incluye *pandas* 3) |
| 25 | 3. Tipos de variables |
| 15 | 4. Sesgos de recolección |
| 10 | 5. Ejercicio guiado (diccionario de datos) y cierre |

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* 🟢 **Ejercicio básico:** lo que todos deben dominar hoy.
* 🔴 **Reto (opcional):** para quien termina antes o quiere profundizar. Si no da tiempo en clase, se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.

## 0. Bienvenida y reglas del curso.

**Diploma:** se obtiene con **calificación final ≥ 70** y **asistencia de al menos 7 de 8 sesiones** (80%; se permite como máximo una falta).

| Evaluación | Se publica | Cierra | Composición |
|---|---|---|---|
| Evaluación semanal 1 (Módulos I y II) | jue 22 oct, 8:30 pm | dom 25 oct, 11:59 pm | Cuestionario 40 + *notebook* 60 |
| Evaluación semanal 2 (Módulos III y IV) | jue 29 oct, 8:30 pm | dom 1 nov, 11:59 pm | Cuestionario 30 + Proyecto integrador 70 |

* **Calificación final = (E1 + E2) / 2.**
* Entregas tardías: hasta 48 h con 10% de penalización; después ya no se califican.
* Los *notebooks* 01 a 06 son **práctica sin calificación**, pero son la mejor preparación para las evaluaciones.

**Arranque del *Codespace*:** en el repositorio del curso, botón verde **Code** → pestaña **Codespaces** → **Create codespace on main**. La primera vez tarda unos minutos porque instala `requirements.txt`. Después se abre este *notebook* y se elige el kernel **Python 3.12**.

### Preparación del entorno.

Se importarán las bibliotecas con sus alias convencionales, se ubicará la carpeta `data/` (que ya trae los CSV del curso, sin necesidad de internet) y se fijará la semilla `401` para que los resultados aleatorios sean reproducibles.

* La siguiente celda importará las bibliotecas y definirá `DATA` y `rng`.

In [1]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)

* La siguiente celda mostrará las versiones instaladas. El curso usa *pandas* 3.

In [2]:
{"NumPy": np.__version__, "pandas": pd.__version__, "seaborn": sns.__version__}

{'NumPy': '2.5.4', 'pandas': '3.0.6', 'seaborn': '0.13.2'}

---
## 1. *NumPy* esencial (25 min).

[*NumPy*](https://numpy.org) es la base de casi todo el ecosistema científico de *Python*: *pandas*, *SciPy*, *statsmodels* y *Matplotlib* lo usan por dentro. Su objeto central es el **arreglo** (`ndarray`):

* **Homogéneo:** todos los elementos tienen el mismo `dtype` (`int64`, `float64`, `bool`, …).
* **De tamaño fijo y contiguo en memoria**, lo que permite operaciones **vectorizadas** escritas en C.

### 1.1 Arreglos y `dtype`.

* La siguiente celda creará el arreglo `masas` con la masa corporal (g) de 8 pingüinos.

In [3]:
masas_lista = [3750, 3800, 3250, 3450, 3650, 3625, 4675, 3475]
masas = np.array(masas_lista)
masas

array([3750, 3800, 3250, 3450, 3650, 3625, 4675, 3475])

* La siguiente celda mostrará el `dtype`, la forma y el número de dimensiones de `masas`.

In [4]:
masas.dtype, masas.shape, masas.ndim

(dtype('int64'), (8,), 1)

**Nota:** si se mezclan enteros y flotantes, *NumPy* promueve todo a `float64`. Como `np.nan` es un flotante, un arreglo con faltantes no puede ser `int64`.

* La siguiente celda mostrará esa promoción de tipos.

In [5]:
np.array([1, 2.5, 3]).dtype, np.array([3750, np.nan, 3250]).dtype

(dtype('float64'), dtype('float64'))

### 1.2 Vectorización.

Operar sobre el arreglo completo, sin bucles explícitos, es más legible y mucho más rápido.

* La siguiente celda convertirá las masas a kilogramos con una sola operación vectorizada.

In [6]:
masas / 1000

array([3.75 , 3.8  , 3.25 , 3.45 , 3.65 , 3.625, 4.675, 3.475])

* La siguiente celda calculará la media, la desviación estándar muestral (`ddof=1`) y el máximo.

In [7]:
masas.mean(), masas.std(ddof=1).round(1), masas.max()

(np.float64(3709.375), np.float64(428.6), np.int64(4675))

* La siguiente celda simulará un millón de masas para comparar tiempos.

In [8]:
grande = rng.normal(4200, 800, size=1_000_000)
grande_lista = grande.tolist()

* La siguiente celda medirá el tiempo de un bucle de *Python* y el de la operación vectorizada.

In [9]:
%timeit -n 3 -r 3 [x / 1000 for x in grande_lista]
%timeit -n 3 -r 3 grande / 1000

39.7 ms ± 9.58 ms per loop (mean ± std. dev. of 3 runs, 3 loops each)
626 μs ± 136 μs per loop (mean ± std. dev. of 3 runs, 3 loops each)


### 1.3 *Broadcasting*.

Cuando se operan arreglos de formas distintas, *NumPy* "estira" virtualmente el más pequeño si las dimensiones son compatibles (iguales, o una de ellas vale 1, comparando de derecha a izquierda).

* La siguiente celda creará una matriz 4×3: filas = pingüinos; columnas = largo del pico, profundidad del pico y largo de la aleta (mm).

In [10]:
medidas = np.array([
    [39.1, 18.7, 181],
    [39.5, 17.4, 186],
    [46.5, 17.9, 192],
    [50.0, 15.2, 218],
])
medidas.shape

(4, 3)

* La siguiente celda calculará la media de cada columna, un arreglo de forma `(3,)`.

In [11]:
medias_columna = medidas.mean(axis=0)
medias_columna

array([ 43.775,  17.3  , 194.25 ])

* La siguiente celda restará las medias a cada fila: `(4, 3) - (3,)` se resuelve por *broadcasting*.

In [12]:
(medidas - medias_columna).round(2)

array([[ -4.67,   1.4 , -13.25],
       [ -4.27,   0.1 ,  -8.25],
       [  2.73,   0.6 ,  -2.25],
       [  6.23,  -2.1 ,  23.75]])

* La siguiente celda centrará por **fila**, para lo cual se necesita una columna `(4, 1)` creada con `np.newaxis`.

In [13]:
medias_fila = medidas.mean(axis=1)[:, np.newaxis]
(medidas - medias_fila).shape

(4, 3)

### 1.4 Máscaras booleanas.

Una comparación produce un arreglo de `True`/`False` que sirve para **filtrar** y para **contar** (`True` vale 1).

* La siguiente celda creará la máscara `pesados`.

In [14]:
pesados = masas > 3600
pesados

array([ True,  True, False, False,  True,  True,  True, False])

* La siguiente celda filtrará con la máscara y contará cuántos valores la cumplen y en qué proporción.

In [15]:
masas[pesados], pesados.sum(), pesados.mean()

(array([3750, 3800, 3650, 3625, 4675]), np.int64(5), np.float64(0.625))

* La siguiente celda combinará condiciones con `&` (y). Cada condición va entre paréntesis.

In [16]:
masas[(masas > 3400) & (masas < 3800)]

array([3750, 3450, 3650, 3625, 3475])

* La siguiente celda usará `np.where` para asignar una etiqueta elemento a elemento.

In [17]:
np.where(masas > 3600, "pesado", "ligero")

array(['pesado', 'pesado', 'ligero', 'ligero', 'pesado', 'pesado',
       'pesado', 'ligero'], dtype='<U6')

### 🟢 Ejercicio 1 · Máscaras y faltantes.

Con el arreglo `masa_g` (la masa de **todos** los pingüinos del *dataset*, incluidos los `NaN`):
1. Conviértelo a kilogramos en `masa_kg`.
2. Calcula la media **ignorando faltantes** (pista: `np.nanmean`). ¿Qué pasa si usas `.mean()`?
3. Cuenta cuántos pingüinos pesan más de 4.5 kg y qué proporción representan **entre los que tienen dato**.

* La siguiente celda extraerá la columna `body_mass_g` como arreglo de *NumPy*.

In [18]:
masa_g = pd.read_csv(os.path.join(DATA, "penguins.csv"))["body_mass_g"].to_numpy()
masa_g[:6], masa_g.dtype, masa_g.shape

(array([3750., 3800., 3250.,   nan, 3450., 3650.]), dtype('float64'), (344,))

* La siguiente celda muestra una solución posible.

In [19]:
masa_kg = masa_g / 1000
con_dato = ~np.isnan(masa_kg)
mas_de_45 = masa_kg[con_dato] > 4.5
{
    "media con .mean()": masa_kg.mean(),
    "media con np.nanmean()": np.nanmean(masa_kg).round(3),
    "más de 4.5 kg": int(mas_de_45.sum()),
    "proporción entre los que tienen dato": mas_de_45.mean().round(3),
}

{'media con .mean()': np.float64(nan),
 'media con np.nanmean()': np.float64(4.202),
 'más de 4.5 kg': 115,
 'proporción entre los que tienen dato': np.float64(0.336)}

**✅ Respuesta sugerida:**

`.mean()` devuelve `nan`: un solo faltante contamina el resultado. `np.nanmean()` da ≈ 4.202 kg. Hay 115 pingüinos de más de 4.5 kg, el 33.6% de los 342 con dato. Si se dividiera entre los 344 (incluidos los `NaN`) se subestimaría la proporción.

### 🟢 Ejercicio 2 · Estandarizar con *broadcasting*.

Estandariza cada **columna** de `medidas` (puntaje *z*: restar la media y dividir entre la desviación estándar de su columna) **sin bucles**. Verifica que las medias resultantes sean ≈ 0 y las desviaciones ≈ 1.

* La siguiente celda muestra una solución posible.

In [20]:
z = (medidas - medidas.mean(axis=0)) / medidas.std(axis=0, ddof=1)
z.round(3), z.mean(axis=0).round(10), z.std(axis=0, ddof=1).round(10)

(array([[-0.872,  0.934, -0.805],
        [-0.797,  0.067, -0.501],
        [ 0.508,  0.4  , -0.137],
        [ 1.161, -1.401,  1.443]]),
 array([0., 0., 0.]),
 array([1., 1., 1.]))

### 🔴 Reto 1 · Matriz de distancias sin bucles.

Calcula la matriz 4×4 de **distancias euclidianas** entre todos los pares de pingüinos de `z` usando solo *broadcasting* (pista: `z[:, None, :] - z[None, :, :]` tiene forma `(4, 4, 3)`). ¿Qué par de pingüinos es el más parecido?

* La siguiente celda muestra una solución posible.

In [21]:
diferencias = z[:, None, :] - z[None, :, :]
distancias = np.sqrt((diferencias ** 2).sum(axis=-1))
sin_diagonal = distancias.copy()
np.fill_diagonal(sin_diagonal, np.inf)
par = np.unravel_index(sin_diagonal.argmin(), sin_diagonal.shape)
distancias.round(2), par

(array([[0.  , 0.92, 1.62, 3.83],
        [0.92, 0.  , 1.4 , 3.13],
        [1.62, 1.4 , 0.  , 2.48],
        [3.83, 3.13, 2.48, 0.  ]]),
 (np.int64(0), np.int64(1)))

---
## 2. *pandas* esencial (35 min).

[*pandas*](https://pandas.pydata.org) agrega a *NumPy* **etiquetas** (índice y nombres de columna) y **columnas de tipos distintos**:

* `Series`: una columna (arreglo 1D + índice).
* `DataFrame`: una tabla en la que cada columna es una `Series` con su propio `dtype`.

### 2.1 *pandas* 3: una gran actualización de arquitectura.

El curso usa **[*pandas* 3.0](https://pandas.pydata.org/docs/whatsnew/v3.0.0.html)** (publicado el 21 de enero de 2026). Es la versión mayor más importante en años y cambia dos cosas de fondo:

| Cambio | *pandas* 2.x | *pandas* 3 |
|---|---|---|
| **Copy-on-Write (CoW)** | Una selección podía ser una vista o una copia, según la operación | **Obligatorio:** todo resultado de una selección o de un método **se comporta como una copia**. La única forma de modificar un objeto es modificarlo directamente (por ejemplo, con `df.loc[...] = ...`). |
| **Tipo de texto por defecto** | `object` (cualquier objeto de *Python*) | **`str`**, un tipo dedicado a texto **respaldado por *PyArrow*** cuando está instalado (en el curso lo está). Solo admite texto o faltantes, y el faltante es siempre `NaN`. |

Además: `SettingWithCopyWarning` desaparece, las fechas leídas de texto se guardan por defecto con resolución de microsegundos (`datetime64[us]`), `groupby` usa `observed=True` por defecto con columnas categóricas y se eliminaron muchas funciones que ya estaban obsoletas. Se requiere *Python* ≥ 3.11 y *NumPy* ≥ 1.26.

> ⚠️ **ADVERTENCIA: incompatibilidad hacia atrás.**
>
> El código escrito para *pandas* 2.x **puede cambiar de comportamiento en silencio o fallar** con *pandas* 3. Los casos más comunes:
> 1. **Asignación encadenada** (`df["col"][i] = valor`, `df[mascara]["col"] = valor`): **ya nunca modifica** el *dataframe* original.
> 2. **Métodos con `inplace=True` sobre una columna** (`df["col"].fillna(0, inplace=True)`): **tampoco modifican** el original.
> 3. **Comprobaciones de tipo de texto** como `df["col"].dtype == object` o `select_dtypes(include="object")`: el texto ahora es `str`.
> 4. **Asignar un valor que no es texto** a una columna `str` (por ejemplo, un número): **lanza `TypeError`**.
>
> Los tutoriales, respuestas de foros y código generado antes de 2026 suelen asumir *pandas* 2. Las celdas siguientes ejecutan cada caso. Para migrar código propio, la recomendación oficial es pasar primero a *pandas* 2.3, eliminar todas sus advertencias y después actualizar ([guía de migración del tipo `str`](https://pandas.pydata.org/docs/user_guide/migration-3-strings.html), [guía de *Copy-on-Write*](https://pandas.pydata.org/docs/user_guide/copy_on_write.html)).

* La siguiente celda creará un *dataframe* pequeño para los ejemplos.

In [22]:
ejemplo = pd.DataFrame({"especie": ["Adelie", "Gentoo", "Chinstrap"],
                        "masa": [3750.0, np.nan, 3700.0]})
ejemplo.dtypes

especie        str
masa       float64
dtype: object

**Ejemplo 1: el texto ya no es `object`.** La columna `especie` es de tipo `str` y está respaldada por *PyArrow*; la comparación con `object` que funcionaba en *pandas* 2 ahora devuelve `False`.

* La siguiente celda comparará el `dtype` de la columna de texto con `object` y con la forma recomendada.

In [23]:
{
    "dtype": repr(ejemplo["especie"].dtype),
    "arreglo interno": type(ejemplo["especie"].array).__name__,
    "dtype == object (estilo pandas 2)": ejemplo["especie"].dtype == object,
    "pd.api.types.is_string_dtype (recomendado)": pd.api.types.is_string_dtype(ejemplo["especie"]),
}

{'dtype': '<StringDtype(na_value=nan)>',
 'arreglo interno': 'ArrowStringArray',
 'dtype == object (estilo pandas 2)': False,
 'pd.api.types.is_string_dtype (recomendado)': True}

**Ejemplo 2: asignación encadenada.** En *pandas* 2 a veces modificaba el original (con `SettingWithCopyWarning`); en *pandas* 3 **nunca** lo hace y emite `ChainedAssignmentError`. La celda captura la advertencia para mostrarla.

* La siguiente celda intentará cambiar la masa del primer pingüino con asignación encadenada.

In [24]:
copia = ejemplo.copy()
with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter("always")
    copia["masa"][0] = 9999
copia.loc[0, "masa"], [type(a.message).__name__ for a in avisos]

(np.float64(3750.0), ['ChainedAssignmentError'])

* La siguiente celda hará la misma asignación de la forma correcta, en un solo paso con `loc`.

In [25]:
copia.loc[0, "masa"] = 9999
copia.loc[0, "masa"]

np.float64(9999.0)

**Ejemplo 3: `inplace=True` sobre una columna.** El método actúa sobre una copia intermedia, así que el faltante sigue ahí.

* La siguiente celda intentará rellenar el faltante con `inplace=True` sobre la columna.

In [26]:
copia = ejemplo.copy()
with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter("always")
    copia["masa"].fillna(0, inplace=True)
copia["masa"].isna().sum(), [type(a.message).__name__ for a in avisos]

(np.int64(1), ['ChainedAssignmentError'])

* La siguiente celda mostrará la forma recomendada: reasignar la columna.

In [27]:
copia["masa"] = copia["masa"].fillna(0)
copia["masa"].isna().sum()

np.int64(0)

**Ejemplo 4: una columna `str` solo admite texto.** En *pandas* 2 la columna `object` aceptaba cualquier objeto; ahora la asignación falla.

* La siguiente celda intentará guardar un número en la columna de texto y mostrará el error.

In [28]:
try:
    copia.loc[0, "especie"] = 5
    resultado = "se asignó"
except TypeError as error:
    resultado = f"TypeError: {error}"
resultado

"TypeError: Invalid value '5' for dtype 'str'. Value should be a string or missing value, got 'int' instead."

**Ejemplo 5: una selección se comporta como copia.** Modificar `serie` no altera `copia`, aunque por dentro *pandas* comparta memoria hasta el momento de escribir (*copy-on-write*).

* La siguiente celda modificará una columna extraída y comprobará que el *dataframe* no cambió.

In [29]:
copia = ejemplo.copy()
serie = copia["masa"]
serie.iloc[2] = 0.0
serie.iloc[2], copia.loc[2, "masa"]

(np.float64(0.0), np.float64(3700.0))

**Regla práctica para todo el curso:** para modificar un *dataframe* se usa **`df.loc[filas, columnas] = valor`** o se **reasigna** (`df["col"] = ...`, `df = df.assign(...)`). Para revisar si una columna es texto se usa `pd.api.types.is_string_dtype`.

### 2.2 `Series` y `DataFrame`.

* La siguiente celda creará una `Series` con índice de letras y calculará su media; `mean()` ignora los `NaN` por defecto.

In [30]:
serie_masas = pd.Series([3750, 3800, np.nan], index=["a", "b", "c"], name="body_mass_g")
serie_masas["b"], serie_masas.mean()

(np.float64(3800.0), np.float64(3775.0))

* La siguiente celda creará un *dataframe* desde un diccionario.

In [31]:
mini = pd.DataFrame({
    "species": ["Adelie", "Gentoo", "Chinstrap"],
    "body_mass_g": [3750, 5000, 3700],
})
mini

,species,body_mass_g
0,Adelie,3750
1,Gentoo,5000
2,Chinstrap,3700


### 2.3 Lectura y primer vistazo: el *dataset* `penguins`.

**Palmer Penguins** (Gorman, Williams y Fraser, 2014): medidas de 344 pingüinos de tres especies en tres islas del archipiélago Palmer (Antártida), registradas en las temporadas 2007 a 2009 por el programa *Palmer Station LTER*. Es el *dataset* principal de las sesiones 1 a 6.

| Columna | Descripción |
|---|---|
| `species` | Especie: Adelie, Chinstrap, Gentoo |
| `island` | Isla: Biscoe, Dream, Torgersen |
| `bill_length_mm` | Largo del pico (culmen), mm |
| `bill_depth_mm` | Profundidad del pico, mm |
| `flipper_length_mm` | Largo de la aleta, mm |
| `body_mass_g` | Masa corporal, g |
| `sex` | Sexo (en este CSV: `MALE`/`FEMALE`) |

* La siguiente celda leerá `penguins.csv` y mostrará las primeras filas.

In [32]:
pinguinos = pd.read_csv(os.path.join(DATA, "penguins.csv"))
pinguinos.head()

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


* La siguiente celda mostrará la forma del *dataframe*.

In [33]:
pinguinos.shape

(344, 7)

**Nota:** `info()` es lo primero que hay que revisar: tipos, conteo de no nulos y memoria. Observa que las columnas de texto aparecen como `str`, no como `object`.

* La siguiente celda mostrará el resumen de `info()`.

In [34]:
pinguinos.info()

<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            344 non-null    str    
 1   island             344 non-null    str    
 2   bill_length_mm     342 non-null    float64
 3   bill_depth_mm      342 non-null    float64
 4   flipper_length_mm  342 non-null    float64
 5   body_mass_g        342 non-null    float64
 6   sex                333 non-null    str    
dtypes: float64(4), str(3)
memory usage: 24.9 KB


* La siguiente celda mostrará `describe(include="all")`, que también resume las columnas de texto.

In [35]:
pinguinos.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
species,344,3,Adelie,152,NaN,NaN,NaN,NaN,NaN,NaN,NaN
island,344,3,Biscoe,168,NaN,NaN,NaN,NaN,NaN,NaN,NaN
bill_length_mm,342.0,NaN,NaN,NaN,43.92193,5.459584,32.1,39.225,44.45,48.5,59.6
bill_depth_mm,342.0,NaN,NaN,NaN,17.15117,1.974793,13.1,15.6,17.3,18.7,21.5
flipper_length_mm,342.0,NaN,NaN,NaN,200.915205,14.061714,172.0,190.0,197.0,213.0,231.0
body_mass_g,342.0,NaN,NaN,NaN,4201.754386,801.954536,2700.0,3550.0,4050.0,4750.0,6300.0
sex,333,2,MALE,168,NaN,NaN,NaN,NaN,NaN,NaN,NaN


* La siguiente celda contará los pingüinos de cada especie.

In [36]:
pinguinos["species"].value_counts(dropna=False)

species
Adelie       152
Gentoo       124
Chinstrap     68
Name: count, dtype: int64

### 2.4 Selección: `[]`, `loc` e `iloc`.

* `df["col"]` devuelve una columna (`Series`); `df[["c1", "c2"]]`, varias (`DataFrame`).
* `df.loc[filas, columnas]` selecciona por **etiqueta** o máscara booleana. El rango `a:b` **incluye** `b`.
* `df.iloc[filas, columnas]` selecciona por **posición** entera. El rango `a:b` **excluye** `b`.

* La siguiente celda seleccionará las filas con etiqueta 0, 1 y 2 de dos columnas usando `loc`.

In [37]:
pinguinos.loc[0:2, ["species", "body_mass_g"]]

,species,body_mass_g
0,Adelie,3750.0
1,Adelie,3800.0
2,Adelie,3250.0


* La siguiente celda seleccionará las posiciones 0 y 1 de las columnas 0 y 5 usando `iloc`.

In [38]:
pinguinos.iloc[0:2, [0, 5]]

,species,body_mass_g
0,Adelie,3750.0
1,Adelie,3800.0


### 2.5 Filtros.

* La siguiente celda contará los Gentoo de más de 5.5 kg con una máscara booleana.

In [39]:
len(pinguinos[(pinguinos["species"] == "Gentoo") & (pinguinos["body_mass_g"] > 5500)])

28

* La siguiente celda usará `isin` para quedarse con dos islas.

In [40]:
pinguinos[pinguinos["island"].isin(["Dream", "Torgersen"])].shape

(176, 7)

* La siguiente celda usará `query`, una alternativa más legible.

In [41]:
pinguinos.query("species == 'Chinstrap' and flipper_length_mm >= 205")

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
181,Chinstrap,Dream,52.8,20.0,205.0,4550.0,MALE
189,Chinstrap,Dream,52.0,20.7,210.0,4800.0,MALE
191,Chinstrap,Dream,53.5,19.9,205.0,4500.0,MALE
192,Chinstrap,Dream,49.0,19.5,210.0,3950.0,MALE
199,Chinstrap,Dream,49.0,19.6,212.0,4300.0,MALE
212,Chinstrap,Dream,51.9,19.5,206.0,3950.0,MALE
215,Chinstrap,Dream,55.8,19.8,207.0,4000.0,MALE
218,Chinstrap,Dream,50.8,19.0,210.0,4100.0,MALE


### 2.6 Crear columnas con `assign` y encadenar métodos.

`assign` devuelve un **nuevo** *dataframe*, sin modificar el original, y permite escribir el análisis como una receta de pasos encadenados. Con *Copy-on-Write* este estilo es eficiente: *pandas* evita copias innecesarias.

* La siguiente celda creará `body_mass_kg`, filtrará Biscoe, elegirá columnas y ordenará por masa.

In [42]:
(
    pinguinos
    .assign(body_mass_kg=lambda d: d["body_mass_g"] / 1000)
    .query("island == 'Biscoe'")
    .loc[:, ["species", "sex", "body_mass_kg"]]
    .sort_values("body_mass_kg", ascending=False)
    .head()
)

,species,sex,body_mass_kg
237,Gentoo,MALE,6.30
253,Gentoo,MALE,6.05
337,Gentoo,MALE,6.00
297,Gentoo,MALE,6.00
299,Gentoo,MALE,5.95


**Nota:** *pandas* 3 introduce además `pd.col("columna")` para escribir expresiones sin `lambda`, por ejemplo `assign(body_mass_kg=pd.col("body_mass_g") / 1000)`. En el curso se usará `lambda` porque funciona en cualquier versión y es lo que se encuentra en la mayoría de la documentación.

### 🟢 Ejercicio 3 · Explorar con *pandas*.

1. ¿Cuántos pingüinos hay de cada especie **en cada isla**? (pista: `pd.crosstab`). ¿Qué especie vive en las tres islas?
2. Con `loc`, obtén `species`, `island` y `body_mass_g` de los Gentoo con masa mayor a 5,000 g, ordenados de mayor a menor masa. ¿Cuántos son?

* La siguiente celda muestra una solución posible.

In [43]:
pd.crosstab(pinguinos["island"], pinguinos["species"])

species,Adelie,Chinstrap,Gentoo
island,,,
Biscoe,44,0,124
Dream,56,68,0
Torgersen,52,0,0


* La siguiente celda muestra una solución posible del inciso 2.

In [44]:
gentoo_5k = (
    pinguinos
    .loc[(pinguinos["species"] == "Gentoo") & (pinguinos["body_mass_g"] > 5000),
         ["species", "island", "body_mass_g"]]
    .sort_values("body_mass_g", ascending=False)
)
len(gentoo_5k), gentoo_5k.head()

(61,
     species  island  body_mass_g
 237  Gentoo  Biscoe       6300.0
 253  Gentoo  Biscoe       6050.0
 297  Gentoo  Biscoe       6000.0
 337  Gentoo  Biscoe       6000.0
 331  Gentoo  Biscoe       5950.0)

**✅ Respuesta sugerida:**

Adelie es la única especie presente en las tres islas; Gentoo solo vive en Biscoe y Chinstrap solo en Dream. Hay 61 Gentoo de más de 5,000 g.

### 🟢 Ejercicio 4 · Nuevas variables con `assign`.

Crea, en un solo `assign`, las columnas `bill_ratio = bill_length_mm / bill_depth_mm` y `body_mass_kg`. Muestra los 5 pingüinos con mayor `bill_ratio`. ¿De qué especie son? ¿Por qué crees que pasa?

* La siguiente celda muestra una solución posible.

In [45]:
(
    pinguinos
    .assign(
        bill_ratio=lambda d: d["bill_length_mm"] / d["bill_depth_mm"],
        body_mass_kg=lambda d: d["body_mass_g"] / 1000,
    )
    .nlargest(5, "bill_ratio")
    .loc[:, ["species", "island", "bill_length_mm", "bill_depth_mm", "bill_ratio", "body_mass_kg"]]
)

,species,island,bill_length_mm,bill_depth_mm,bill_ratio,body_mass_kg
307,Gentoo,Biscoe,51.3,14.2,3.612676,5.30
240,Gentoo,Biscoe,50.2,14.3,3.510490,5.70
253,Gentoo,Biscoe,59.6,17.0,3.505882,6.05
220,Gentoo,Biscoe,46.1,13.2,3.492424,4.50
283,Gentoo,Biscoe,54.3,15.7,3.458599,5.65


**✅ Respuesta sugerida:**

Los 5 con mayor `bill_ratio` son **Gentoo**: su pico es largo y poco profundo (delgado), por lo que el cociente largo/profundidad es alto. La razón resume la **forma** del pico, no su tamaño.

### 🔴 Reto 2 · Asignación encadenada en *pandas* 3.

El siguiente código intenta poner en `NaN` la masa de los pingüinos de Torgersen. Con *pandas* 3 **no modifica `copia`** y emite `ChainedAssignmentError`. Explica por qué y corrígelo con `loc` en una sola instrucción. Comprueba el resultado contando los faltantes por isla.

```python
copia = pinguinos.copy()
copia[copia["island"] == "Torgersen"]["body_mass_g"] = np.nan
```

* La siguiente celda muestra una solución posible.

In [46]:
copia = pinguinos.copy()
copia.loc[copia["island"] == "Torgersen", "body_mass_g"] = np.nan
copia["body_mass_g"].isna().groupby(copia["island"]).sum()

island
Biscoe        1
Dream         0
Torgersen    52
Name: body_mass_g, dtype: int64

**✅ Respuesta sugerida:**

El original hace **dos** operaciones: `copia[mascara]` produce un objeto nuevo que, con *Copy-on-Write*, siempre se comporta como copia, y `["body_mass_g"] = ...` modifica ese objeto intermedio, que se descarta. Con `loc`, selección y asignación son **una sola** operación sobre `copia`. Torgersen queda con 52 faltantes (todos sus pingüinos) y Biscoe conserva el único que ya tenía.

---
## 3. Tipos de variables (25 min).

El **tipo estadístico** de una variable decide qué resúmenes, gráficas y pruebas tienen sentido. No es lo mismo que el `dtype` de *pandas*, pero conviene que coincidan.

| Tipo estadístico | Subtipo | Ejemplo | Resumen típico | `dtype` sugerido |
|---|---|---|---|---|
| **Numérica** | Continua | `body_mass_g`, `fare` | media, mediana, desv. est., IQR | `float64` |
| | Discreta (conteos) | `sibsp`, `size` (tips) | igual + frecuencias | `int64` / `Int64` (*nullable*) |
| **Categórica** | Nominal (sin orden) | `species`, `island`, `sex` | frecuencias, moda | `category` (o `str`) |
| | Ordinal (con orden) | `class` (First > Second > Third) | frecuencias, mediana, percentiles | `category` ordenada |
| **Fecha / tiempo** | | fecha de muestreo | rango, frecuencia por periodo | `datetime64[us]` |
| **Identificador** | | `id_muestra` | ninguno (no se promedia) | `str` |

**Advertencia:** son trampas comunes los códigos numéricos que en realidad son categorías (`pclass` = 1, 2, 3; códigos postales), los conteos guardados como `float` solo porque hay faltantes y los `bool` que en realidad son una categoría binaria.

### 3.1 `category`: nominal y ordinal.

En *pandas* 3 el texto ya se lee como `str`, que es eficiente. `category` sigue siendo útil para fijar el **conjunto de valores válidos** y, sobre todo, para variables **ordinales**.

* La siguiente celda convertirá `species` en categoría nominal y mostrará sus categorías.

In [47]:
tipos = pinguinos.copy()
tipos["species"] = tipos["species"].astype("category")
tipos["species"].cat.categories.tolist()

['Adelie', 'Chinstrap', 'Gentoo']

* La siguiente celda creará la variable ordinal `talla` a partir de la masa, solo para ilustrar.

In [48]:
tipos["talla"] = pd.cut(
    tipos["body_mass_g"],
    bins=[0, 3500, 4500, np.inf],
    labels=["chico", "mediano", "grande"],
    ordered=True,
)
tipos["talla"].value_counts(sort=False)

talla
chico       78
mediano    149
grande     115
Name: count, dtype: int64

* La siguiente celda comparará categorías ordenadas: cuenta las tallas de *mediano* en adelante.

In [49]:
(tipos["talla"] >= "mediano").sum()

np.int64(264)

* La siguiente celda comparará la memoria de `island` como `str` (respaldado por *PyArrow*) y como `category`.

In [50]:
{
    "str": pinguinos["island"].memory_usage(deep=True),
    "category": pinguinos["island"].astype("category").memory_usage(deep=True),
}

{'str': 4980, 'category': 521}

### 3.2 Tipos *nullable*: enteros con faltantes.

`flipper_length_mm` y `body_mass_g` son enteros en el origen, pero *pandas* los lee como `float64` porque tienen `NaN`. Los tipos *nullable* (`Int64`, `boolean`, con mayúscula) representan el faltante como `pd.NA` sin cambiar el tipo.

* La siguiente celda convertirá `flipper_length_mm` a `Int64`.

In [51]:
pinguinos["flipper_length_mm"].astype("Int64").head()

0     181
1     186
2     195
3    <NA>
4     193
Name: flipper_length_mm, dtype: Int64

* La siguiente celda mostrará los tipos que propone `convert_dtypes()`.

In [52]:
pinguinos.convert_dtypes().dtypes

species               string
island                string
bill_length_mm       Float64
bill_depth_mm        Float64
flipper_length_mm      Int64
body_mass_g            Int64
sex                   string
dtype: object

### 3.3 Fechas.

`penguins` no trae fechas, así que se usará un pequeño registro ficticio de salidas de campo. En *pandas* 3, `pd.to_datetime` sobre texto produce por defecto `datetime64[us]` (microsegundos) en lugar de `datetime64[ns]`.

* La siguiente celda creará el registro con las fechas como texto.

In [53]:
salidas = pd.DataFrame({
    "fecha": ["2007-11-11", "2008-11-09", "2009-11-18", "2009-12-01"],
    "isla": ["Torgersen", "Biscoe", "Dream", "Biscoe"],
})
salidas.dtypes

fecha    str
isla     str
dtype: object

* La siguiente celda convertirá `fecha` a `datetime` y derivará la temporada, el día de la semana y los días transcurridos.

In [54]:
salidas["fecha"] = pd.to_datetime(salidas["fecha"])
salidas = salidas.assign(
    temporada=salidas["fecha"].dt.year,
    dia_semana=salidas["fecha"].dt.day_name(),
    dias_desde_inicio=(salidas["fecha"] - salidas["fecha"].min()).dt.days,
)
salidas

,fecha,isla,temporada,dia_semana,dias_desde_inicio
0,2007-11-11,Torgersen,2007,Sunday,0
1,2008-11-09,Biscoe,2008,Sunday,364
2,2009-11-18,Dream,2009,Wednesday,738
3,2009-12-01,Biscoe,2009,Tuesday,751


* La siguiente celda mostrará el `dtype` resultante de la fecha.

In [55]:
salidas["fecha"].dtype

dtype('<M8[us]')

### 🟢 Ejercicio 5 · Clasifica las variables de `titanic`.

Clasifica por tipo estadístico estas columnas: `survived`, `pclass`, `sex`, `age`, `sibsp`, `fare`, `embarked`, `class`, `deck`, `alone`. Anota también qué `dtype` les asignó *pandas* y si lo cambiarías. Responde en la celda de respuesta (puedes usar una tabla).

* La siguiente celda leerá `titanic.csv` y mostrará sus tipos.

In [56]:
titanic = pd.read_csv(os.path.join(DATA, "titanic.csv"))
titanic.dtypes

survived         int64
pclass           int64
sex                str
age            float64
sibsp            int64
parch            int64
fare           float64
embarked           str
class              str
who                str
adult_male        bool
deck               str
embark_town        str
alive              str
alone             bool
dtype: object

**✅ Respuesta sugerida:**

| Columna | Tipo estadístico | `dtype` leído | `dtype` sugerido |
|---|---|---|---|
| `survived` | Categórica binaria (nominal) | `int64` | `category` o `bool` |
| `pclass` | Categórica **ordinal** (codificada con números) | `int64` | `category` ordenada |
| `sex` | Categórica nominal | `str` | `category` |
| `age` | Numérica continua | `float64` | `float64` |
| `sibsp` | Numérica discreta (conteo) | `int64` | `int64` |
| `fare` | Numérica continua | `float64` | `float64` |
| `embarked` | Categórica nominal | `str` | `category` |
| `class` | Categórica ordinal (First > Second > Third) | `str` | `category` ordenada |
| `deck` | Categórica (ordinal según la posición en el barco; muchos faltantes) | `str` | `category` |
| `alone` | Categórica binaria | `bool` | `bool` |

`pclass` y `class` son **la misma información**: hay redundancia.

### 🔴 Reto 3 · Ordinal en la práctica.

Convierte `titanic["class"]` en categoría **ordenada** de forma que `First` sea la **más alta**. Después calcula, con una comparación (`>=`), la proporción de pasajeros que viajaban en `Second` o superior, y la tasa de supervivencia por clase en el orden correcto.

* La siguiente celda muestra una solución posible.

In [57]:
orden_clase = pd.CategoricalDtype(["Third", "Second", "First"], ordered=True)
clase = titanic["class"].astype(orden_clase)
(clase >= "Second").mean().round(3), titanic["survived"].groupby(clase).mean().round(3)

(np.float64(0.449),
 class
 Third     0.242
 Second    0.473
 First     0.630
 Name: survived, dtype: float64)

---
## 4. Sesgos de recolección (15 min).

Antes de calcular nada hay que preguntar **¿cómo llegaron estos datos aquí?** Ningún método estadístico corrige un sesgo de recolección que no se reconoce.

| Sesgo | Qué ocurre | Ejemplo |
|---|---|---|
| **Selección** | La muestra no representa a la población de interés | Medir pingüinos solo en las colonias más accesibles |
| **Supervivencia** | Solo se observa a quienes pasaron un filtro | Analizar solo empresas que siguen operando; aviones que regresaron |
| **No respuesta** | Quienes no responden difieren de quienes sí | Encuesta de ingreso que contestan menos los de ingresos altos |
| **Medición** | El instrumento o el procedimiento distorsiona el valor | Báscula descalibrada; preguntas que inducen la respuesta; redondeos |

**Ejemplo:** se simulará un sesgo de **selección**: ¿qué pasaría si solo se hubiera muestreado la isla Biscoe?

* La siguiente celda comparará la proporción de especies en todas las islas y solo en Biscoe.

In [58]:
completo = pinguinos.dropna(subset=["body_mass_g"])
solo_biscoe = completo[completo["island"] == "Biscoe"]
pd.DataFrame({
    "todas las islas": completo["species"].value_counts(normalize=True),
    "solo Biscoe": solo_biscoe["species"].value_counts(normalize=True),
}).fillna(0).round(3)

,todas las islas,solo Biscoe
species,,
Adelie,0.442,0.263
Chinstrap,0.199,0.000
Gentoo,0.360,0.737


* La siguiente celda comparará la masa media en ambos casos.

In [59]:
{"todas las islas": completo["body_mass_g"].mean().round(0),
 "solo Biscoe": solo_biscoe["body_mass_g"].mean().round(0)}

{'todas las islas': np.float64(4202.0), 'solo Biscoe': np.float64(4716.0)}

La masa media cambia unos 500 g solo por **dónde** se muestreó: Biscoe concentra a los Gentoo, que son los más grandes. La muestra no está mal, pero **las conclusiones solo valen para la población que representa**.

### 🟢 Ejercicio 6 · Identifica el sesgo.

Para cada caso, indica el sesgo principal y una forma de mitigarlo:
1. Una *app* de *fitness* concluye que sus usuarios bajan de peso analizando solo a quienes siguen activos después de 6 meses.
2. Una encuesta telefónica en horario laboral sobre hábitos de consumo.
3. Un estudio de propinas (`tips`) registrado por **un solo mesero** en **un solo restaurante**.
4. Un sensor de temperatura que redondea hacia abajo al entero más cercano.

**✅ Respuesta sugerida:**

1. **Supervivencia** (o abandono): quienes no bajaron de peso tienden a dejar la *app*. Mitigar: analizar a todos los que empezaron y registrar los abandonos.
2. **Selección / no respuesta:** subrepresenta a quien trabaja en ese horario. Mitigar: varios horarios y canales, ponderar por perfil.
3. **Selección:** los resultados describen a ese mesero y ese restaurante, no a "los clientes" en general. Mitigar: muestrear varios restaurantes y meseros, o limitar las conclusiones.
4. **Medición** (sistemático): subestima en promedio unos 0.5 °C. Mitigar: calibrar o corregir el sesgo conocido.

---
## 5. Ejercicio guiado: diccionario de datos de `penguins` (10 min + práctica).

Un **diccionario de datos** documenta cada columna: qué es, de qué tipo es, en qué unidades está y qué valores puede tomar. Es el primer entregable de cualquier EDA serio y se usará en el proyecto integrador.

### 🟢 Ejercicio 7a · Construye el diccionario.

Crea un *dataframe* `diccionario` con una fila por columna de `pinguinos` y las columnas: `columna`, `tipo_estadistico`, `dtype_pandas`, `unidad`, `valores_posibles` (rango observado si es numérica; lista de categorías si es de texto) y `n_faltantes`.

*Pista:* `dtype_pandas`, `valores_posibles` y `n_faltantes` pueden calcularse con código; `tipo_estadistico` y `unidad` son conocimiento del dominio y se escriben a mano.

* La siguiente celda muestra una solución posible.

In [60]:
tipo_estadistico = {
    "species": "categórica nominal",
    "island": "categórica nominal",
    "bill_length_mm": "numérica continua",
    "bill_depth_mm": "numérica continua",
    "flipper_length_mm": "numérica continua (registrada como entero)",
    "body_mass_g": "numérica continua (registrada como entero)",
    "sex": "categórica nominal (binaria)",
}
unidad = {"bill_length_mm": "mm", "bill_depth_mm": "mm", "flipper_length_mm": "mm", "body_mass_g": "g"}

def valores_posibles(serie):
    if pd.api.types.is_numeric_dtype(serie):
        return f"{serie.min():g} – {serie.max():g}"
    return ", ".join(sorted(serie.dropna().unique()))

diccionario = pd.DataFrame({
    "columna": pinguinos.columns,
    "tipo_estadistico": [tipo_estadistico[c] for c in pinguinos.columns],
    "dtype_pandas": [str(pinguinos[c].dtype) for c in pinguinos.columns],
    "unidad": [unidad.get(c, "—") for c in pinguinos.columns],
    "valores_posibles": [valores_posibles(pinguinos[c]) for c in pinguinos.columns],
    "n_faltantes": pinguinos.isna().sum().to_numpy(),
})
diccionario

,columna,tipo_estadistico,dtype_pandas,unidad,valores_posibles,n_faltantes
0,species,categórica nominal,str,—,"Adelie, Chinstrap, Gentoo",0
1,island,categórica nominal,str,—,"Biscoe, Dream, Torgersen",0
2,bill_length_mm,numérica continua,float64,mm,32.1 – 59.6,2
3,bill_depth_mm,numérica continua,float64,mm,13.1 – 21.5,2
4,flipper_length_mm,numérica continua (registrada como entero),float64,mm,172 – 231,2
5,body_mass_g,numérica continua (registrada como entero),float64,g,2700 – 6300,2
6,sex,categórica nominal (binaria),str,—,"FEMALE, MALE",11


### 🟢 Ejercicio 7b · Convierte los `dtypes`.

Crea `pinguinos_tipados` donde `species`, `island` y `sex` sean `category`, y `flipper_length_mm` y `body_mass_g` sean `Int64` (*nullable*). Verifica con `info()` y compara la memoria usada contra el original.

* La siguiente celda muestra una solución posible.

In [61]:
pinguinos_tipados = pinguinos.astype({
    "species": "category",
    "island": "category",
    "sex": "category",
    "flipper_length_mm": "Int64",
    "body_mass_g": "Int64",
})
pinguinos_tipados.info()

<class 'pandas.DataFrame'>
RangeIndex: 344 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   species            344 non-null    category
 1   island             344 non-null    category
 2   bill_length_mm     342 non-null    float64 
 3   bill_depth_mm      342 non-null    float64 
 4   flipper_length_mm  342 non-null    Int64   
 5   body_mass_g        342 non-null    Int64   
 6   sex                333 non-null    category
dtypes: Int64(2), category(3), float64(2)
memory usage: 12.7 KB


* La siguiente celda comparará la memoria total de ambos *dataframes* (en bytes).

In [62]:
{"original": int(pinguinos.memory_usage(deep=True).sum()),
 "tipado": int(pinguinos_tipados.memory_usage(deep=True).sum()) if pinguinos_tipados is not None else None}

{'original': 25465, 'tipado': 12978}

### 🟢 Ejercicio 7c · Pregunta de sesgo.

Los datos se tomaron en **solo tres islas** del archipiélago Palmer y en **tres temporadas de anidación** (2007 a 2009). **Nota:** la versión del CSV que se usa no incluye la columna del año; el dato viene de la documentación del *dataset*.

¿Qué sesgo podría tener esta muestra? ¿Para qué población son válidas las conclusiones y qué **no** se debería afirmar?

**✅ Respuesta sugerida:**

* **Sesgo de selección / cobertura espacial:** tres islas no representan a todas las colonias de cada especie. Además, las especies no están repartidas igual (Gentoo solo en Biscoe, Chinstrap solo en Dream), así que **especie e isla están confundidas**: una diferencia "entre islas" puede ser en realidad una diferencia entre especies, y viceversa.
* **Cobertura temporal:** tres temporadas consecutivas pueden coincidir con condiciones particulares (alimento, clima, hielo marino) y no capturan tendencias de largo plazo.
* **Momento del muestreo / medición:** se mide durante la anidación; la masa corporal cambia a lo largo del ciclo reproductivo.
* **Supervivencia:** solo se miden adultos que llegaron a reproducirse.

Las conclusiones valen para **pingüinos reproductores de esas colonias en 2007 a 2009**. No se debería generalizar a "todos los Adelie de la Antártida" ni atribuir diferencias a la isla sin considerar la especie.

### 🔴 Reto 4 · Muestreo por conveniencia simulado.

Simula que un equipo solo pudo medir a los **100 pingüinos más fáciles de atrapar**, con probabilidad de captura mayor para los más ligeros (proporcional a `1 / body_mass_g**3`). Con `rng.choice(..., p=..., replace=False)` toma esa muestra y compara la masa media y la proporción de especies con las del *dataset* completo. ¿De qué signo es el sesgo?

* La siguiente celda muestra una solución posible.

In [63]:
base = pinguinos.dropna(subset=["body_mass_g"]).reset_index(drop=True)
peso = 1 / base["body_mass_g"].to_numpy() ** 3
indices = rng.choice(len(base), size=100, replace=False, p=peso / peso.sum())
muestra = base.iloc[indices]
pd.DataFrame({
    "completo": base["species"].value_counts(normalize=True),
    "conveniencia": muestra["species"].value_counts(normalize=True),
}).round(3), base["body_mass_g"].mean().round(0), muestra["body_mass_g"].mean().round(0)

(           completo  conveniencia
 species                          
 Adelie        0.442          0.58
 Chinstrap     0.199          0.26
 Gentoo        0.360          0.16,
 np.float64(4202.0),
 np.float64(3807.0))

**✅ Respuesta sugerida:**

El sesgo es **negativo**: la muestra por conveniencia subestima la masa media y además cambia la composición de especies (hay menos Gentoo).

---
## ✅ Cierre.

**Hoy vimos:**
* *NumPy*: arreglos homogéneos, vectorización, *broadcasting* y máscaras booleanas.
* *pandas* 3: *Copy-on-Write* obligatorio, el tipo `str` respaldado por *PyArrow* y por qué el código de *pandas* 2 puede comportarse distinto.
* *pandas*: `read_csv`, `info`, `describe`, `loc`/`iloc`, filtros, `query` y `assign` encadenado.
* Tipos de variables y su `dtype`: `str`, `category` (nominal/ordinal), tipos *nullable* (`Int64`) y fechas.
* Sesgos de recolección: selección, supervivencia, no respuesta y medición.

**Práctica (sin calificación, en Classroom):** terminar los ejercicios 🟢 pendientes y, si es posible, un reto 🔴.

**Próxima sesión (mar 20 oct):** calidad de datos: duplicados, inconsistencias y valores faltantes (`02_calidad_y_faltantes.ipynb`).

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>